### 1. Process kml file into site_constants .csv

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

site_fn = 'data/by_glacier/kahiltna/site_constants.csv'

gdf = gpd.read_file('C:/Users/cvw30/Research/data/nps_glacier_coords.kml') 
gdf['lon'] = gdf.geometry.x
gdf['lat'] = gdf.geometry.y
gdf = gdf.loc[np.abs(gdf['lon'] - -151.2) < 0.5, ['lon','lat','Name']] 
gdf = gdf.set_index('Name')

df = pd.read_csv(site_fn, index_col='site')
for point in gdf.index:
    point_up = point.upper()
    if point_up not in df.index:
        df.loc[point_up] = np.zeros_like(df.iloc[0]) * np.nan 
        df.loc[point_up, 'lat'] = gdf.loc[point, 'lat']
        df.loc[point_up, 'lon'] = gdf.loc[point, 'lon']

df.to_csv(site_fn)

### 2. Process WGMS file for a certain glacier for all sites

In [26]:
import os 
import pandas as pd
import numpy as np

glac_name = 'taku'
min_n_points = 10
site_fn = f'../../data/by_glacier/{glac_name}/site_constants.csv'

wgms_fn = '../../../data/wgms/data/mass_balance_point.csv'
df = pd.read_csv(wgms_fn)
df = df.loc[df['glacier_name'] == glac_name.upper()]
df['begin_date'] = np.array([pd.to_datetime(d.replace('-','/')) for d in df['begin_date']])
df = df.loc[df['begin_date'] > pd.to_datetime('2000-01-01')]

df = (df.groupby('original_id')
      .filter(lambda g: g['balance'].notna().sum() >= min_n_points))

sites = df['original_id'].unique()
site_df = pd.read_csv(site_fn, index_col='site')
for site in sites:
      site_lat_mean = np.nanmean(df.loc[df['original_id'] == site, 'latitude'])
      site_lon_mean = np.nanmean(df.loc[df['original_id'] == site, 'longitude'])
      site_elev_mean = np.nanmean(df.loc[df['original_id'] == site, 'elevation'])
      site_up = site.upper()
      if site_up not in site_df.index:
            site_df.loc[site_up] = np.zeros_like(site_df.iloc[0]) * np.nan 
            site_df.loc[site_up, 'lat'] = site_lat_mean
            site_df.loc[site_up, 'lon'] = site_lon_mean
            site_df.loc[site_up, 'elevation'] = site_elev_mean.round(1)

site_df.to_csv(site_fn)
site_df

,lat,lon,elevation,sky_view,slope,aspect
site,,,,,,
center,58.651000,-134.278000,1367.0,0.998792,1.292431,153.22237
D,58.684868,-134.348341,NaN,NaN,NaN,NaN
C,58.628860,-134.230867,NaN,NaN,NaN,NaN
B,58.560575,-134.139942,NaN,NaN,NaN,NaN
C161,58.632957,-134.418588,1485.0,NaN,NaN,NaN
TKG5,58.603878,-134.180019,995.4,NaN,NaN,NaN
TKG4,58.634407,-134.237005,1116.2,NaN,NaN,NaN
TKG6,58.668817,-134.282719,1185.9,NaN,NaN,NaN
DG1,58.617588,-134.130047,1016.7,NaN,NaN,NaN
